In [46]:
%run common_setup.ipynb

#### MatchFactory matches two srings using exact, acronym, short and fuzzy

In [47]:
import glob
import re
from unidecode import unidecode
from polyfuzz import PolyFuzz
from polyfuzz.models import RapidFuzz, TFIDF, EditDistance, Embeddings
from flair.embeddings import TransformerWordEmbeddings, WordEmbeddings
from jellyfish import jaro_winkler_similarity

class MatchFactory:

    def __init__(self):
        self.factory = {
            'exact': self._exact_matcher,
            'acronym': self._acronym_matcher,
            'short': self._short_matcher,
            'fuzzy': self._fuzzy_matcher
            }
        return

    def match_factory(self, from_list, to_list):
        self.from_list = from_list
        self.to_list = to_list
        match = {}
        for match_type in self.factory.keys():
            print(f'{len(from_list) = } {len(to_list) = }')
            match  |= self.factory[match_type](from_list, to_list)
            # print(match)
            from_list = [from_name for from_name in from_list if from_name not in match.keys()]
            to_list = [to_name for to_name in to_list if to_name not in match.values()]
            if not from_list or not to_list:
                return match
        return match

    def _exact_matcher(self, from_list, to_list):
        exact_match = []
        exact_match.extend(item for item in from_list if item in to_list)
        # print(f'{len(exact_match) = }/{len(from_list) = }')
        return dict(zip(exact_match, exact_match))

    def _acronym_matcher(self, from_list, to_list):
        acronym_match = {}
        for from_name in from_list:
            if m := re.search(r'([A-Z]+?){3,}', from_name):
                match = m[0]
                if match not in ['USA', 'UK']:
                    # print(f'{from_name = } {match = }')
                    for to_name in to_list:
                        if match == to_name: # or match in to_name:
                            # print(f'== {from_name = } {match = } {to_name = }')
                            acronym_match[from_name] = to_name
        return acronym_match

    def _short_matcher(self, from_list, to_list):
        to_dict = dict(zip(to_list, self.tidy_list(to_list)))
        short_match = {}
        for from_name, from_short in zip(from_list, self.tidy_list(from_list)):
            # print(f'{from_name = } {from_short = }')
            for to_name, to_short in to_dict.items():
                # print(f'{to_name = } {to_short = }')
                if from_short in to_short:
                    short_match[from_name] = to_name
                    break
        return short_match

    def _fuzzy_matcher(self, from_list, to_list):
        tfidf = TFIDF(n_gram_range=(3,3), min_similarity=0.5, model_id="TF-IDF")
        rapid_fuzz = RapidFuzz(n_jobs=1, score_cutoff=0.8, model_id='RapidFuzz')
        matchers = [tfidf, rapid_fuzz]
        model = PolyFuzz(matchers)
        model.match(from_list, to_list) #self.tidy_list(from_list), to_list)
        match = pd.concat(list(model.get_matches().values()), axis=0).sort_values('Similarity', ascending=False).drop_duplicates(subset=['From'])
        # print(match.tail(32))
        return {f: t for f, t, s in zip(match.From, match.To, match.Similarity) if s > 0.8 and t != None}

    def tidy_list(self, in_list):
        out_list = []
        for item in in_list:
            parts = [part.strip().lower() for part in unidecode(item).split(' ')]
            out_list.append(' '.join([p for p in parts if p not in ['the', 'of', '&', 'and', 'de', '-', 'universite', 'universidade', 'universitat', 'university']])) #'university'
        return out_list

#### Matches the jouranls and institutions in Domingo's lists to JCR and OpenAlex 

In [48]:
class MatchDomingoSourceInstitutionSamples(SetUp, MatchFactory):

    def __init__(self):
        super().__init__()
        return
    
    def special_issn(self):
        special_issn = {'economic research-ekonomska istrazivanja': ['1331-677X'], # '1848-9664'],
                                'finanzarchiv': 	['0015-2218'],
                                'journal of the knowledge economy': 	['1868-7865'], #, '1868-7873'],
                                'latin american economic review': 	['2196-436X']} #, '2198-3526']}
        df = pd.DataFrame.from_dict(special_issn, orient='index').reset_index()
        df.columns = ['Journal name', 'ISSN']
        print(f'SPECIAL matches {df.shape = }\n{df.head()}')
        self.db.sql("CREATE OR REPLACE TABLE memory.specials AS (SELECT * FROM df)")
        self.db.sql("SELECT count(*) FROM memory.specials").show()
        return
    
    def extract_jcr(self):
        df_list = []
        for file in glob.glob('../DATA/WOS_JCR/*.xlsx'):
            df_list.append(pd.read_excel(file, skiprows=2))
        df = pd.concat(df_list).drop_duplicates()
        for row in df.itertuples():
            if not isinstance(row.ISSN, str):
                df.at[row.Index, 'ISSN'] = row.eISSN
        print(f'{df.shape = }\n{df.head()}')
        self.db.sql("CREATE OR REPLACE TABLE memory.wos_jcr AS (SELECT * FROM df)")
        self.db.sql("SELECT * FROM memory.wos_jcr").show()
        self.db.sql("COPY memory.wos_jcr TO '../DATA/wos_jcr.csv' (HEADER, DELIMITER ',')")
        return

    def extract_journals_institutions(self):
        df_dd = pd.read_excel('../DATA/eco_bus_inst_journal_scores.xlsx', sheet_name=None)
        for kind in ['journals', 'institutions']:
            df = self._tidy_journals(df=df_dd[kind]) if kind == 'journals' else df_dd[kind]
            df = df.drop(columns=['Unnamed: 0'])
            print(f'{kind = } {df.shape = }\n{df.head()}')
            self.db.sql(f"CREATE OR REPLACE TABLE memory.{kind} AS (SELECT * FROM df)")
            self.db.sql(f"SELECT count(*) FROM memory.{kind}").show()
        return

    def _tidy_journals(self, df=None):
        df = df.replace(to_replace='(?i)^Economics and Philosophy$', value='economics & philosophy', regex=True) 
        df = df.replace(to_replace='(?i)^revista de historia economica$', value='Revista de Historia Economica-Journal of Iberian and Latin American Economic History', regex=True)
        df = df.replace(to_replace='(?i)^spanish journal of finance and accounting-revista espanola de financiacion y contabilida$', value=
                                 'spanish journal of finance and accounting-revista espanola de financiacion y contabilidad', regex=True)
        df = df.replace(to_replace='(?i)BUSINESS ETHICS-A EUROPEAN REVIEW', value='BUSINESS ETHICS THE ENVIRONMENT & RESPONSIBILITY'.capitalize(), regex=True)
        return df
    
    def source_matcher(self):
        sql = """
            CREATE OR REPLACE TABLE project.jcr_matches AS
            SELECT DISTINCT ON (journal)
                    jcr_journal,
                    ISSN AS issn,
                    journal,
                    acr,
                    pub,
                    journal_score
                FROM journals j
                LEFT JOIN 
                    (SELECT "Journal name" as jcr_journal,
                            ISSN
                        FROM memory.wos_jcr
                     UNION
                     SELECT "Journal name" as jcr_journal,
                            ISSN
                        FROM memory.specials s
                    )
                ON lower(journal) = lower(jcr_journal)
                -- WHERE ISSN IS NULL
                ORDER BY journal
            """
        self.db.sql(sql)
        self.db.sql("SELECT count(*) FROM project.jcr_matches").show()
        self.db.sql("SELECT * FROM project.jcr_matches").show()
        return
    
    def institution_matcher(self):

        df = self.db.sql("SELECT * FROM memory.institutions").df() #.iloc[:64]
        
        oa = self._extract_oa_institutions()
        oa_dict = dict(zip(oa.institution_name, oa.institution_id))
        print(f'{oa.shape = }\n{oa.head()}')
        print(oa.loc[oa.institution_name.str.contains('INSEAD')].head(99))

        from_list = df.institution.to_list()
        to_list = oa.institution_name.to_list()
        matched = MatchFactory().match_factory(from_list, to_list)
        
        df.insert(0, 'institution_id', [oa_dict.get(matched.get(from_name)) for from_name in df.institution])
        df.insert(1, 'institution_name', [matched.get(from_name) for from_name in df.institution])
        print(df[df.institution_id.isnull()].head())
        self.db.sql("CREATE OR REPLACE TABLE project.institutions_matched AS SELECT * FROM df")
        self.db.sql("SELECT count(*) FROM project.institutions_matched").show()
        self.db.sql("SELECT * FROM project.institutions_matched").show()
        return

    def _extract_oa_institutions(self):
        sql = """
            SELECT id AS institution_id,
                    unnest(all_names) AS institution_name,
                    ror,
                    works_count 
            FROM
                (SELECT id, ror, works_count,
                        list(display_name) OVER (PARTITION BY id) AS display_name_list,
                        list_concat(display_name_list, display_name_alternatives, display_name_acronyms) AS all_names
                FROM institutions.institutions)
                WHERE works_count > 500
            """
        return self.db.sql(sql).df()

In [49]:
class CompareDomingoSourceInstitutionStatus(SetUp):

    def __init__(self):
        super().__init__()
        return

    def compare_sources(self):
        domingo = self.db.sql("SELECT * FROM project.jcr_matches").df()
        print(f'{domingo.shape = }\n{domingo.head()}')
        return

#### This cell matches Domingo's C and T (and CT) lists to authors in the OpenAlex corpus
#### It also creates the X and Y samples from OpenAlex  

- Extract Domingo's list and ensure that the names are normalised
- Match Domingo' Names to OpenAlex display_names
- Allocate the T, C, TC, X and Y samples

In [50]:
class MatchDomingoAuthorSample(SetUp, MatchFactory):

    def __init__(self):
        super().__init__()
        return    

    def extract_sample(self):
        sample = pd.read_excel('../DATA/researchers_results_total_average_influence.xlsx').iloc[:, :10]
        sample[['first', 'middle', 'last', 'fullname']] = [normalise_name(n) for n in sample.Research_Profile]
        print(f'{sample.shape = }\n{sample.head()}')
        sample = sample.sort_values('HCP', ascending=False).reset_index(drop=True)
        print(sample[sample.duplicated(keep=False)].head(32))
        self.db.sql("CREATE OR REPLACE TABLE project.domingo_sample_original AS SELECT * FROM sample")
        self.sample = self.db.sql("SELECT * FROM project.domingo_sample_original").df()
        print(f'{sample.shape = }\n{sample.head()}')
        return
    
    def factory_matcher(self):  # sourcery skip: identity-comprehension
        authors = self.db.sql("""SELECT id AS author_id, unnest(display_name_alternatives) AS author_name 
                                    FROM authors.authors WHERE works_count > 100 ORDER BY works_count DESC""").df()
        print(f'{authors.shape = }\n{authors.head()}')
        sample = self.sample
        from_list = sample.fullname.to_list()
        to_dict = {n: id for n, id in zip(authors.author_name, authors.author_id)} #normalise_name(n)[-1]: id 
        to_list = list(to_dict.keys())
        print(f'{to_list[:8]}\n{from_list[:8]}')
        matched = MatchFactory().match_factory(from_list, to_list)
        sample.insert(1, 'author_id', [matched.get(from_name) for from_name in self.sample.fullname])
        sample.insert(2, 'author_name', [to_dict.get(matched.get(from_name)) for from_name in sample.fullname])
        print(f'{sample.shape = }\n{sample.head()}')
        self.db.sql("CREATE OR REPLACE TABLE project.domingo_sample_matched AS SELECT * FROM sample")
        self.db.sql("SELECT * FROM project.domingo_sample_matched").show()
        self.sample = sample
        return

    def match_sample(self):
        sql = """   
            -- MATCH Domingo's EconBus list to OpenAlex
            -- matches on full name and first intial/last name
            -- finds 1280 matches (many duplicates)
            -- finds 293 distinct matches from endogenous authors
            -- finds 26 non-matching WOS names, no duplicates.
            -- there are 15 "complex" names matched by hand
            -- 
            -- ========================================
            CREATE OR REPLACE TABLE project.candidates AS
            WITH 
                match_endogenous_CTE AS
                (SELECT a.author_id,
                        a.orcid,
                        a.works_count,
                        PUB,
                        PUB/a.works_count AS works_fraction,
                        a.cited_by_count,
                        CIT,
                        CIT/a.cited_by_count AS cite_fraction,
                        a.author_name,
                        a.fullname AS oa_fullname,
                        aa.topics[1].field.display_name AS field,
                        Research_Profile,
                        "Group",
                        "class",
                        d."first",
                        d.middle,
                        d."last",
                        d.fullname,
                    FROM project.domingo_sample_original d
                    LEFT JOIN project.authors_full a
                    ON list_contains(display_name_alternatives, d.fullname) OR list_contains(display_name_alternatives, concat(d.first[1],'. ', d.last))
                    LEFT JOIN project.authors aa
                    USING (author_id)
                    ORDER BY d.fullname, a.works_count DESC
                    ),
                filtered_endogenous_CTE AS
                    (SELECT m.*
                    FROM match_endogenous_CTE m
                    WHERE list_contains(['Agricultural and Biological Sciences', 'Arts and Humanities', 'Environmental Science',
                                            'Energy', 'Psychology', -- 'Decision Sciences', 'Mathematics'
                                            'Biochemistry, Genetics and Molecular Biology', 'Computer Science',
                                            'Chemical Engineering', 'Earth and Planetary Sciences', 'Engineering', 
                                            'Materials Science' , 'Medicine', 'Neuroscience', 'Physics and Astronomy'], field) = false
                    ),
                endogenous_matched_CTE AS
                    (SELECT DISTINCT ON (Research_Profile)
                            * 
                    FROM filtered_endogenous_CTE
                    ORDER BY Research_Profile, works_count DESC
                    ),
                exogenous_match_CTE AS  
                    (SELECT oa.id AS author_id,
                            oa.orcid,
                            oa.works_count,
                            PUB,
                            PUB/oa.works_count AS works_fraction,
                            oa.cited_by_count,
                            CIT,
                            CIT/oa.cited_by_count AS cite_fraction,
                            oa.display_name AS author_name,
                            oa.display_name AS oa_fullname,
                            oa.topics[1].field.display_name AS field,
                            Research_Profile,
                            "Group",
                            "class",
                            "first",
                            middle,
                            "last",
                            fullname
                        FROM match_endogenous_CTE e
                        LEFT JOIN authors.authors oa
                        ON list_contains(oa.display_name_alternatives, e.fullname) OR list_contains(oa.display_name_alternatives, concat(e.first[1],'. ', e.last))
                        WHERE (e.author_id IS NULL) AND 
                            (field is NULL OR list_contains(['Physics and Astronomy', 'Earth and Planetary Sciences', 
                                            'Biochemistry, Genetics and Molecular Biology', 'Medicine', 
                                            'Veterinary', 'Engineering', 'Neuroscience', 'Nursing',
                                            'Health Professions', 'Materials Science'], field) = false) 
                        ORDER BY Research_Profile, oa.works_count DESC
                    ),
                difficult_names_CTE AS
                (SELECT oa.id AS author_id,
                        oa.orcid,
                        oa.works_count,
                        PUB,
                        PUB/oa.works_count AS works_fraction,
                        oa.cited_by_count,
                        CIT,
                        CIT/oa.cited_by_count AS cite_fraction,
                        oa.display_name AS author_name,
                        oa.display_name AS oa_fullname,
                        oa.topics[1].field.display_name AS field,
                        Research_Profile,
                        "Group",
                        "class",
                        "first",
                        middle,
                        "last",
                        fullname
                    FROM '/home/lc/Projects/EconomicsBusiness/DATA/difficult_name_matches.csv'
                    LEFT JOIN project.domingo_sample_original d
                    USING (Research_Profile)
                    LEFT JOIN authors.authors oa
                    ON id = author_id
                ),
                openalex_kind_CTE AS
                (SELECT author_id,
                        orcid,
                        works_count,
                        NULL AS PUB,
                        NULL AS works_fraction,
                        cited_by_count,
                        NULL AS CIT,
                        NULL AS cite_fraction,
                        author_name,
                        fullname AS oa_fullname,
                        topics[1].field.display_name AS field,
                        NULL AS Research_Profile,
                        'X' AS "Group",
                        NULL AS "class",
                        NULL AS "first",
                        NULL AS middle,
                        NULL AS "last",
                        NULL AS fullname
                        -- row_number() OVER (ORDER BY works_count DESC) AS row_count
                    FROM project.authors_full a
                    WHERE list_contains((SELECT list(author_id) FROM project.candidates GROUP BY ALL), author_id) = false
                            AND list_contains(['Economics, Econometrics and Finance'], field) --'Decision Sciences',  'Social Sciences'], field)
                    LIMIT 150
                    )

                        -- SELECT 'openalex' AS kind,
                        --         *
                        --   FROM openalex_kind_CTE

            SELECT *
            FROM
                (SELECT 'endogenous' AS kind,
                            *
                        FROM endogenous_matched_CTE
                    UNION  
                        SELECT DISTINCT ON (Research_Profile)
                            'exogenous' AS kind,
                                *
                        FROM exogenous_match_CTE
                        WHERE author_id IS NOT NULL
                    UNION
                        SELECT DISTINCT ON (Research_Profile)
                                'unmatched' AS kind,
                                *
                        FROM exogenous_match_CTE
                        WHERE author_id IS NULL
                    UNION
                        SELECT DISTINCT ON (Research_Profile)
                                'matched_difficult' AS kind,
                                *
                        FROM difficult_names_CTE
                    UNION
                        SELECT 'openalex' AS kind,
                                *
                        FROM openalex_kind_CTE
                    )
            """
        self.db.sql(sql)  #.show()
        return

    def load_sample(self):
        print('load matched sample together with author data from OA')
        sample= self.db.sql("SELECT * FROM project.candidates").df().sort_values('Research_Profile').reset_index(drop=True)
        print(f'{sample.shape = } {sample['kind'].unique() = }\n{sample.head()}\n{sample.loc[[s is None for s in sample.kind], :].head()}')
        with pd.ExcelWriter('../DATA/domingo_sample_match.xlsx') as writer:
            sample.to_excel(writer, index=False, sheet_name='all_data')
            sample.query("kind == 'endogenous'").to_excel(writer, index=False, sheet_name='endogenous')
            sample.query("kind == 'exogenous'").to_excel(writer, index=False, sheet_name='exogneous')
            sample.query("kind == 'unmatched'").to_excel(writer, index=False, sheet_name='unmatched')
            sample.query("kind == 'matched_difficult'").to_excel(writer, index=False, sheet_name='unmatched')
            sample.query("kind == 'openalex'").to_excel(writer, index=False, sheet_name='openalex')
            sample.loc[[k is None for k in sample["kind"]], :].to_excel(writer, index=False, sheet_name='unmatched_')
            sample.loc[[k is not None for k in sample["author_id"]], :].to_excel(writer, index=False, sheet_name='best_case')
        return

#### This class builds CSV files for Domingo made up from the OA data

In [51]:
class ETLForDomingo(SetUp):

    def __init__(self):
        super().__init__()
        self.assemble_tables = {}
        return
    
    def match_table(self):
        sql = """  
            -- MATCH Domingo's EconBus list to OpenAlex
            -- ========================================
            SELECT *
            FROM project.candidates
        """
        sample = self.db.sql(sql).df()
        sample = sample.sort_values(['Group', 'Research_Profile'], ascending=[False, True]).reset_index(drop=True)
        print(f'{sample.shape = }\n{sample.head()}')
        self.assemble_tables |= {'Domingo_matched': sample}
        return
    
    def works_table(self):
        sql = """
            -- ETL works for Domingo
            -- =====================
            SELECT id as work_id,
                    doi, 
                    title, 
                    publication_year,
                    fwci,
                    cited_by_count,
                    referenced_works_count,
                    "primary_location.source".display_name AS source_name,
                    "primary_location.source".issn_l AS issn,
                    "primary_location.source".host_organization_name AS publisher,
                    "biblio.volume" AS volume,
                FROM project.raw
            """
        works = self.db.sql(sql).df()
        works = works.sort_values(['publication_year', 'fwci'], ascending=[True, False]).reset_index(drop=True)
        print(f'{works.shape = }\n{works.head()}')
        self.assemble_tables |= {'works': works}
        return
    
    def authorships_table(self):
        sql = """
            -- ETL FOR Domingo authors and institutions
            -- ========================================
            SELECT work_id,
                    author_name,
                    author_id,
                    institution.display_name AS institution_name,
                    institution.id AS institution_id,
                    institution.country_code AS country_code
            FROM
                (SELECT work_id,
                    authorship.author.display_name as author_name,
                    authorship.author.id AS author_id,
                    unnest(authorship.institutions) AS institution,
                FROM 
                    (SELECT id AS work_id,
                        unnest(authorships) AS authorship
                    FROM project.raw
                    )
                )
                """
        authorships = self.db.sql(sql).df()
        print(f"{authorships.shape = }\n{authorships.head()}")
        self.assemble_tables |= {'authorships': authorships}
        return
    
    def references_table(self):
        sql = """
            -- ETL FOR Domingo reference_list
            -- ==============================
            SELECT id AS work_id,
                    referenced_works
                FROM project.raw
            """
        references = self.db.sql(sql).df()
        print(f'{references.shape = }\n{references.head()}')
        self.assemble_tables |= {'references': references}
        return
    
    def citations_table(self):
        sql = """
            -- ETL FOR Domingo citation_list
            -- ==============================
            SELECT cited_id,
                    list(citer_id) AS citing_works_list
            FROM 
                (SELECT id AS citer_id,
                    unnest(referenced_works) AS cited_id
                FROM project.raw
                )
            GROUP BY ALL
            """
        citations = self.db.sql(sql).df()
        print(f'{citations.shape = }\n{citations.head()}')
        self.assemble_tables |= {'citations': citations}
        return
    
    def topics_table(self):
        sql = """ 
            -- ETL FOR Domingo topics
            -- ======================
            SELECT id AS work_id,
                    "primary_topic.display_name" AS topic_name,
                    "primary_topic.subfield".display_name AS subfield_name,
                    "primary_topic.field".display_name AS field_name,
                    "primary_topic.domain".display_name AS domain_name,
                    "primary_topic.score" AS topic_score
            FROM project.raw
            """
        topics = self.db.sql(sql).df()
        print(f'{topics.shape = }\n{topics.head()}')
        self.assemble_tables |= {'topics': topics}
        return


    def load_tables(self):
        for k, v in self.assemble_tables.items():
            with open(f'../DATA/tables_for_Domingo_{k}.csv', 'w') as writer:
                print(f'{k = } {v.shape = }\n{v.head()}')
                v.to_csv(writer, index=False)


In [52]:
def main():

    # mdsis = MatchDomingoSourceInstitutionSamples()
    # mdsis.special_issn()
    # mdsis.extract_jcr()
    # mdsis.extract_journals_institutions()
    # mdsis.source_matcher()
    # mdsis.institution_matcher()
    # mdsis.db.close()

    # cdsis = CompareDomingoSouceInstitutionStatus()
    # cdsis.compare_sources()
    # cdsis.close()

    mdas = MatchDomingoAuthorSample()
    mdas.extract_sample()
    mdas.factory_matcher()
    mdas.db.close()
    
    # tfd = TablesForDomingo()
    # tfd.match_table()
    # tfd.works_table()
    # tfd.authorships_table()
    # tfd.references_table()
    # tfd.citations_table()
    # tfd.topics_table()
    # tfd.load_tables()
    # tfd.db.close()

    return

In [53]:
if __name__ == "__main__":
    main()
    print("DONE")

┌──────────────┬─────────┬──────────────────────┬──────────────────────┬───────────────────────────────────┬───────────┐
│   database   │ schema  │         name         │     column_names     │           column_types            │ temporary │
│   varchar    │ varchar │       varchar        │      varchar[]       │             varchar[]             │  boolean  │
├──────────────┼─────────┼──────────────────────┼──────────────────────┼───────────────────────────────────┼───────────┤
│ authors      │ main    │ authors              │ [id, orcid, displa…  │ [VARCHAR, VARCHAR, VARCHAR, 'VA…  │ false     │
│ institutions │ main    │ institutions         │ [id, ror, display_…  │ [VARCHAR, VARCHAR, VARCHAR, VAR…  │ false     │
│ institutions │ main    │ ror                  │ [name, institution…  │ [VARCHAR, VARCHAR]                │ false     │
│ project      │ main    │ authors              │ [author_id, orcid,…  │ [VARCHAR, VARCHAR, VARCHAR, 'VA…  │ false     │
│ project      │ main    │ autho